# Future Interns - Machine Learning Task 1 (2026)
## Sales & Demand Forecasting for Businesses
**Author**: Future Interns Contributor / Sujan Shetty  
**Topic**: End-to-End Retail Sales Forecasting & Operational Decision Engine  
**Dataset**: Superstore Retail Transaction Dataset  

---
### Business Problem Statement
Companies rely heavily on accurate sales forecasts to:
1. **Plan Inventory**: Determine optimal buffer stocks and reorder points to prevent stockouts.
2. **Manage Cash Flow**: Anticipate future revenue and operational capital requirements.
3. **Optimize Staffing**: Allocate shift workers effectively during demand surges.
4. **Avoid Waste & Capital Lockup**: Minimize holding costs and overstocking markdowns.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src modules can be imported
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_superstore_data, clean_sales_data, aggregate_time_series
from src.features import create_features, get_feature_names
from src.models import ModelArena, TimeSeriesForecaster
from src.evaluator import calculate_metrics, compute_residuals, compute_feature_importance
from src.business_planner import (
    calculate_inventory_plan,
    calculate_cashflow_projections,
    calculate_staffing_plan,
    simulate_scenario,
    generate_executive_summary
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported successfully!')


### 1. Data Ingestion & Missing Value Handling


In [ ]:
# Load and clean raw Superstore data
raw_df = load_superstore_data('../data/superstore_sales.csv')
print(f'Total Transaction Records: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
print(f'Date Range: {raw_df["Order Date"].min().strftime("%Y-%m-%d")} to {raw_df["Order Date"].max().strftime("%Y-%m-%d")}')
raw_df.head(3)


### 2. Time-Series Aggregation (Contiguous Daily Sales)


In [ ]:
# Resample transactions into a contiguous daily time series
daily_ts = aggregate_time_series(raw_df, freq='Daily')
print(f'Contiguous Daily Time-Series Length: {len(daily_ts)} days')
daily_ts.head()


In [ ]:
# Plot daily sales over time
plt.figure(figsize=(14, 5))
plt.plot(daily_ts['ds'], daily_ts['y'], label='Daily Sales ($)', color='#2563eb', alpha=0.6, linewidth=1)
plt.plot(daily_ts['ds'], daily_ts['y'].rolling(14).mean(), label='14-Day Moving Average', color='#f59e0b', linewidth=2)
plt.title('Superstore Historical Daily Sales & Trend', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales Revenue ($)')
plt.legend()
plt.tight_layout()
plt.show()


### 3. Feature Engineering for Time Series


In [ ]:
# Construct calendar, cyclical, lag, and rolling features
featured_df = create_features(daily_ts, date_col='ds', target_col='y', drop_na=True)
feat_cols = get_feature_names(featured_df)
print(f'Engineered {len(feat_cols)} features:')
print(feat_cols)
featured_df.tail(3)


### 4. Temporal Model Arena & Benchmarking


In [ ]:
# Benchmarking multi-model architectures on out-of-time test split
arena = ModelArena(test_days=45)
leaderboard = arena.train_and_benchmark(daily_ts)
print('--- Model Benchmark Leaderboard ---')
display(leaderboard)


### 5. Best Model Evaluation & Holdout Validation


In [ ]:
best_model_name = arena.best_model_name
best_forecaster = arena.get_best_model()
res = arena.results[best_model_name]

test_df = res['test_df']
y_test = test_df['y'].values
y_pred = res['y_pred']

plt.figure(figsize=(14, 5))
plt.plot(test_df['ds'], y_test, label='Actual Holdout Sales', color='#0f172a', linewidth=2)
plt.plot(test_df['ds'], y_pred, label=f'Predicted ({best_model_name})', color='#10b981', linewidth=2, linestyle='--')
plt.title(f'Holdout Evaluation: Actual vs Predicted ({best_model_name})', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()


### 6. Multi-Step Future Forecasting (30 Days Ahead with Confidence Intervals)


In [ ]:
# Generate 30-day recursive future forecast
future_forecast = best_forecaster.forecast_future(daily_ts, horizon_days=30)

plt.figure(figsize=(14, 5))
plt.plot(daily_ts['ds'].tail(60), daily_ts['y'].tail(60), label='Historical Sales (Recent 60 Days)', color='#64748b')
plt.plot(future_forecast['ds'], future_forecast['y_pred'], label='30-Day ML Forecast', color='#10b981', linewidth=2)
plt.fill_between(future_forecast['ds'], future_forecast['y_lower'], future_forecast['y_upper'], color='#10b981', alpha=0.15, label='95% Confidence Band')
plt.title('30-Day Future Sales & Demand Projection', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()


### 7. Business Planning & Operational Insights


In [ ]:
# Compute supply chain inventory recommendations
inv_plan = calculate_inventory_plan(future_forecast, lead_time_days=5, service_level=0.95)
print('--- Inventory Replenishment Plan ---')
print(f'Average Daily Demand: {inv_plan["avg_daily_demand"]} units/day')
print(f'Required Safety Stock Buffer: {inv_plan["safety_stock"]} units')
print(f'Recommended Reorder Point (ROP): {inv_plan["reorder_point"]} units')
print(f'Optimal Order Batch Size (EOQ): {inv_plan["economic_order_quantity"]} units')
print(f'Critical Surge Days Identified: {inv_plan["total_surge_days"]} days')

# Cash flow forecast
cash_plan = calculate_cashflow_projections(future_forecast)
print('\n--- Financial & Cash Flow Projections ---')
print(f'Expected 30-Day Revenue: ${cash_plan["total_projected_revenue"]:,.2f}')
print(f'Conservative 95% Bound: ${cash_plan["conservative_revenue"]:,.2f}')
print(f'Estimated Operating Gross Profit: ${cash_plan["estimated_gross_profit"]:,.2f}')


### 8. Executive Memo for Store Owners & Stakeholders


In [ ]:
# Generate Plain English Executive Memo
metrics = res['metrics']
memo = generate_executive_summary(daily_ts, future_forecast, best_model_name, metrics, inv_plan, cash_plan)
print(memo)
